# Lab dl_08 · LoRA fine-tuning of your GPT (GPU)

**Goal.** Implement Low-Rank Adaptation ([Hu et al. 2021](https://arxiv.org/abs/2106.09685)), inject it into the attention
projections of the character GPT from dl_05, and fine-tune on a small new text while training < 2 % of the parameters.

### Theory recap
A pretrained weight $W_0\in\mathbb R^{d_{out}\times d_{in}}$ is frozen; we learn a low-rank update
$$h = W_0x + \frac{\alpha}{r}\,BAx,\qquad A\in\mathbb R^{r\times d_{in}},\; B\in\mathbb R^{d_{out}\times r},\; r\ll d.$$
Trainable parameters: $r(d_{in}+d_{out})$ instead of $d_{in}d_{out}$. $A$ is random, $B = 0$ at init, so the adapted model starts
**exactly** at the pretrained one. After training the update can be merged, $W' = W_0 + \frac{\alpha}{r}BA$, so inference costs
nothing extra. The hypothesis behind LoRA: the weight *change* needed for adaptation has low intrinsic rank.

**How this works.** Fill in every cell marked `# TODO` (they currently `raise NotImplementedError`). Run the notebook top to bottom with the working directory set to `labs/`. The last cell runs hidden checks on your functions and records the result in the Learning Center. Do the paper homework for the unlocking skill first — the lab assumes you can do the math by hand.

*This lab is original course material, inspired by Andrej Karpathy's open-source teaching code ([micrograd](https://github.com/karpathy/micrograd), [nn-zero-to-hero](https://github.com/karpathy/nn-zero-to-hero), [nanoGPT](https://github.com/karpathy/nanoGPT), [build-nanogpt](https://github.com/karpathy/build-nanogpt), [minbpe](https://github.com/karpathy/minbpe)). Watching the corresponding lecture after finishing is highly recommended.*

In [ ]:
import math
import os
from dataclasses import dataclass

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- the GPT from lab dl_05 (given; finish dl_05 first if you have not) ----
@dataclass
class GPTConfig:
    vocab_size: int = 65
    block_size: int = 128
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 256
    dropout: float = 0.0

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.c_attn = nn.Linear(cfg.n_embd, 3 * cfg.n_embd)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.c_proj(y.transpose(1, 2).contiguous().view(B, T, C))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd)
        self.c_proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.c_proj(F.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1, self.attn = nn.LayerNorm(cfg.n_embd), CausalSelfAttention(cfg)
        self.ln_2, self.mlp = nn.LayerNorm(cfg.n_embd), MLP(cfg)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        return x + self.mlp(self.ln_2(x))

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte, self.wpe = nn.Embedding(cfg.vocab_size, cfg.n_embd), nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, 0.0, 0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)
    def forward(self, idx, targets=None):
        x = self.drop(self.wte(idx) + self.wpe(torch.arange(idx.shape[1], device=idx.device)))
        for blk in self.blocks:
            x = blk(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None if targets is None else F.cross_entropy(logits.flatten(0, 1), targets.flatten())
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, n, temperature=0.8, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            nxt = torch.multinomial(F.softmax(logits[:, -1] / temperature, -1), 1, generator=generator)
            idx = torch.cat([idx, nxt], 1)
        return idx

def get_batch(data, block_size, batch_size, generator=None, device="cpu"):
    ix = torch.randint(0, len(data) - block_size, (batch_size,), generator=generator)
    x = torch.stack([data[i:i + block_size] for i in ix]); y = torch.stack([data[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

def fit(model, data, steps, lr, batch_size=64, params=None):
    params = [p for p in model.parameters() if p.requires_grad] if params is None else params
    opt = torch.optim.AdamW(params, lr=lr, betas=(0.9, 0.95))
    g, losses = torch.Generator().manual_seed(0), []
    for _ in range(steps):
        x, y = get_batch(data, model.cfg.block_size, batch_size, g, DEVICE)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE == "cuda"):
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step()
        losses.append(loss.item())
    return losses

## 1. The LoRA layer
`LoRALinear(base, r, alpha)`: keep the wrapped layer as `self.base` and freeze it; `self.A` (r, in) with `nn.init.kaiming_uniform_(A, a=math.sqrt(5))`, `self.B` (out, r) zeros; create them on the base layer's device/dtype. `forward(x) = base(x) + (alpha/r) * x @ A.T @ B.T`. `merged_weight()` returns $W_0 + \frac{\alpha}{r}BA$.

`apply_lora(model, target_names, r, alpha)`: freeze **all** parameters of the model, then replace every `nn.Linear` child whose attribute name is in `target_names` (e.g. `"c_attn"`) by a `LoRALinear`. `count_trainable(model)` counts parameters with `requires_grad`. `lora_param_formula(d_in, d_out, r)`.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int = 8, alpha: float = 16.0):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError

    def merged_weight(self):
        # TODO
        raise NotImplementedError


def apply_lora(model, target_names=("c_attn",), r=8, alpha=16.0):
    # TODO
    raise NotImplementedError


def count_trainable(model):
    # TODO
    raise NotImplementedError


def lora_param_formula(d_in, d_out, r):
    # TODO
    raise NotImplementedError

## 2. A pretrained base model
Loads the checkpoint saved by dl_05 (`data/dl05_gpt.pt`); otherwise pretrains the same model for ~1500 steps (about a minute on the GPU).

In [ ]:
from pathlib import Path
text = Path("data/tinyshakespeare.txt").read_text(encoding="utf-8")
if os.path.exists("data/dl05_gpt.pt"):
    ck = torch.load("data/dl05_gpt.pt", map_location=DEVICE)
    stoi = ck["stoi"]; cfg = GPTConfig(**{k: v for k, v in ck["cfg"].items() if k in GPTConfig.__dataclass_fields__})
    base = GPT(cfg).to(DEVICE); base.load_state_dict(ck["model"]); print("loaded dl_05 checkpoint")
else:
    stoi = {c: i for i, c in enumerate(sorted(set(text)))}
    cfg = GPTConfig(vocab_size=len(stoi))
    base = GPT(cfg).to(DEVICE)
    fit(base, torch.tensor([stoi[c] for c in text]), steps=1500, lr=1e-3)
    torch.save({"cfg": cfg.__dict__, "model": base.state_dict(), "stoi": stoi}, "data/dl05_gpt.pt")
itos = {i: c for c, i in stoi.items()}

## 3. A new small text (original, written for this lab)
We fine-tune on a tiny 'play' about a lighthouse keeper — new characters and vocabulary, same alphabet.

In [ ]:
NEW_TEXT = """THE LIGHTHOUSE KEEPER:
I wind the lamp at dusk and watch the water turn to slate.
The gulls go quiet, and the boats come home in twos and threes.
I count them in. I count them twice. I write their names in chalk.

THE FERRYWOMAN:
Keeper, the fog is thick tonight; my lantern barely shows.
Keep your great eye turning, and I shall find the quay.

THE LIGHTHOUSE KEEPER:
It turns, it turns; it never sleeps, and neither, friend, shall I.
Bring salt and bread and news of town, and I will trade you tea.

THE FERRYWOMAN:
The town is loud with market day; the baker burned his rye.
The mayor lost his hat again; the river took it east.
"""

assert set(NEW_TEXT) <= set(stoi), set(NEW_TEXT) - set(stoi)
new_data = torch.tensor([stoi[c] for c in NEW_TEXT])

@torch.no_grad()
def loss_on(model, data, block=64):
    model.eval()
    x = torch.stack([data[i:i + block] for i in range(0, len(data) - block - 1, 16)]).to(DEVICE)
    y = torch.stack([data[i + 1:i + block + 1] for i in range(0, len(data) - block - 1, 16)]).to(DEVICE)
    out = model(x, y)[1].item(); model.train(); return out

print("base model loss on the new text:", loss_on(base, new_data))

## 4. Full fine-tuning vs. LoRA

In [ ]:
import copy
full = copy.deepcopy(base)
full_losses = fit(full, new_data, steps=300, lr=3e-4, batch_size=16)

lora_model = apply_lora(copy.deepcopy(base), target_names=("c_attn",), r=8, alpha=16).to(DEVICE)
n_train, n_total = count_trainable(lora_model), sum(p.numel() for p in lora_model.parameters())
print(f"LoRA trainable: {n_train:,} of {n_total:,} ({100 * n_train / n_total:.2f}%)",
      " formula:", cfg.n_layer * lora_param_formula(cfg.n_embd, 3 * cfg.n_embd, 8))
lora_losses = fit(lora_model, new_data, steps=300, lr=3e-3, batch_size=16)

plt.plot(full_losses, label="full fine-tune"); plt.plot(lora_losses, label="LoRA r=8 on c_attn"); plt.legend(); plt.ylabel("train loss"); plt.show()
print("Shakespeare loss after fine-tuning — base / full / LoRA:",
      [round(loss_on(m, torch.tensor([stoi[c] for c in text[-20000:]])), 3) for m in (base, full, lora_model)])
start = torch.tensor([[stoi["T"]]], device=DEVICE)
print(''.join(itos[i] for i in lora_model.generate(start, 300, generator=torch.Generator(device=DEVICE).manual_seed(0))[0].tolist()))

### Reflect (on paper)
1. Compute by hand the LoRA parameter count for $r = 8$ on `c_attn` in every layer of this model, and for GPT-3 175B with $r=4$ on $W_Q, W_V$.
2. Which forgets more Shakespeare — full fine-tuning or LoRA? Why might a low-rank update act as a regulariser?
3. After merging ($W' = W_0 + \frac{\alpha}{r}BA$), can you still "remove" the adapter? What do you need to keep?

## Grade
Runs the hidden checks (fast, small inputs, fixed seeds) and reports to the local app.

In [ ]:
from lc_client import grade
grade("dl_08_lora", globals())